In [1]:
import json
import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI
from dotenv import load_dotenv
import os

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [3]:
load_dotenv("../.env")

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [4]:
from src.rag import answer_movie_question

/Users/phoothwincho/anaconda3/envs/movie-rag-new/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
#Load Evaluation Questions
with open("../data/retrieval_ground_truth.json") as f:
    evaluation_set = json.load(f)

len(evaluation_set)

30

In [6]:
#generate answer
results = []

for item in tqdm(evaluation_set):

    answer = answer_movie_question(
        item["question"]
    )

    results.append({

        "question": item["question"],

        "expected": item["relevant_movies"],

        "answer": answer

    })

results_df = pd.DataFrame(results)

results_df.head()

  0%|          | 0/30 [00:00<?, ?it/s]

,question,expected,answer
0,A science fiction movie about space exploration,"[Interstellar, The Martian]",### Best match: **Prometheus (2012)**\n- **Gen...
1,A mind-bending science fiction movie,"[Inception, Tenet]",### Best match: **Multiverse (2021)**\n- **Gen...
2,A movie about time travel,"[Back to the Future, Looper]",### Best matches (time travel)\n\n**1) About T...
3,A superhero movie with Marvel characters,"[Avengers: Endgame, Iron Man]",### Best match: **Marvel Studios: Assembling a...
4,A superhero movie from DC,"[The Dark Knight, Man of Steel]",### Best match (DC superhero)\n\n**Justice Lea...


In [7]:
def judge_answer(question, expected, answer):

    prompt = f"""
You are evaluating a Movie Knowledge Assistant.

Question:
{question}

Example Relevant Movies:
{expected}

Assistant Answer:
{answer}

The example relevant movies are ONLY examples.
There may be many other correct movie recommendations.

Do NOT penalize the assistant simply because it recommends
different but equally appropriate movies.

Evaluate the answer based on:

1. Relevance (1-5)

- Does the answer satisfy the user's request?

2. Correctness (1-5)

- Are the movie facts accurate?
- Are the recommendations appropriate?

3. Completeness (1-5)

5 = Recommendation + explanation + genre + director + rating
4 = Missing one item
3 = Missing two items
2 = Very limited information
1 = Almost no useful information

Return ONLY JSON.

{{
    "relevance": 5,
    "correctness": 5,
    "completeness": 5
}}
"""

    response = client.chat.completions.create(

        model="gpt-5.4-nano",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    return response.choices[0].message.content

In [8]:
import json

scores = []

for row in tqdm(results):

    score = judge_answer(

        row["question"],

        row["expected"],

        row["answer"]

    )

    score = json.loads(score)

    scores.append({

        **row,

        **score

    })

judge_df = pd.DataFrame(scores)

judge_df.head()

  0%|          | 0/30 [00:00<?, ?it/s]

,question,expected,answer,relevance,correctness,completeness
0,A science fiction movie about space exploration,"[Interstellar, The Martian]",### Best match: **Prometheus (2012)**\n- **Gen...,5,3,4
1,A mind-bending science fiction movie,"[Inception, Tenet]",### Best match: **Multiverse (2021)**\n- **Gen...,2,1,2
2,A movie about time travel,"[Back to the Future, Looper]",### Best matches (time travel)\n\n**1) About T...,5,4,4
3,A superhero movie with Marvel characters,"[Avengers: Endgame, Iron Man]",### Best match: **Marvel Studios: Assembling a...,3,2,2
4,A superhero movie from DC,"[The Dark Knight, Man of Steel]",### Best match (DC superhero)\n\n**Justice Lea...,5,4,4


In [9]:
judge_df[
    [
        "relevance",
        "correctness",
        "completeness"
    ]
].mean()

relevance       4.433333
correctness     3.133333
completeness    3.733333
dtype: float64

In [10]:
judge_df["overall"] = judge_df[
    [
        "relevance",
        "correctness",
        "completeness"
    ]
].mean(axis=1)

judge_df.head()

,question,expected,answer,relevance,correctness,completeness,overall
0,A science fiction movie about space exploration,"[Interstellar, The Martian]",### Best match: **Prometheus (2012)**\n- **Gen...,5,3,4,4.000000
1,A mind-bending science fiction movie,"[Inception, Tenet]",### Best match: **Multiverse (2021)**\n- **Gen...,2,1,2,1.666667
2,A movie about time travel,"[Back to the Future, Looper]",### Best matches (time travel)\n\n**1) About T...,5,4,4,4.333333
3,A superhero movie with Marvel characters,"[Avengers: Endgame, Iron Man]",### Best match: **Marvel Studios: Assembling a...,3,2,2,2.333333
4,A superhero movie from DC,"[The Dark Knight, Man of Steel]",### Best match (DC superhero)\n\n**Justice Lea...,5,4,4,4.333333


In [11]:
judge_df.to_csv(
    "../data/llm_evaluation.csv",
    index=False
)

In [12]:
summary = pd.DataFrame({

    "Metric":[
        "Average Relevance",
        "Average Correctness",
        "Average Completeness",
        "Overall Score"
    ],

    "Score":[

        judge_df["relevance"].mean(),

        judge_df["correctness"].mean(),

        judge_df["completeness"].mean(),

        judge_df["overall"].mean()

    ]

})

summary

,Metric,Score
0,Average Relevance,4.433333
1,Average Correctness,3.133333
2,Average Completeness,3.733333
3,Overall Score,3.766667


In [ ]:
summary.to_csv(
    "../data/llm_summary.csv",
    index=False
)

: 